In [ ]:
import importlib

import torch, random
import torch.nn.functional as F
import numpy as np
# import accelerate

from lm_eval.__main__ import cli_evaluate
from lm_eval.api.instance import Instance
from lm_eval.api.model import LM
from lm_eval.api.registry import register_model
from tqdm import tqdm

from transformers import AutoTokenizer

from modeling_llada_yukai_06 import LLaDAModelLM
from configs_llada import DiffusionConfig
from components_llada import SimpleLogitsSnapshot
from tools_llada import BlockDiffusionQuotaHelper
from plugins_llada import \
    SaveKVPreviousPlugin_Disabled, SaveKVPreviousPlugin_Enabled,\
    CachePastKVPlugin_Disabled, CachePastKVPlugin_Enabled,\
    CacheAttnPlugin_Disabled, CacheAttnPlugin_Enabled,\
    CacheVOPlugin_Disabled, CacheVOPlugin_Enabled


DTYPE_EVAL = torch.bfloat16

def set_seed(seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
# end

def load_klass(path):
    path_module, name_klass = path.rsplit('.', 1)
    module = importlib.import_module(path_module)
    return getattr(module, name_klass)
# end

class RunModelSemi:

    @ torch.no_grad()
    def run_model(self, model, x, y, config_diffusion, *args, **kwargs):

        '''declare required variables'''
        num_blocks = config_diffusion.num_blocks
        step_per_block = config_diffusion.step_per_block
        size_block = config_diffusion.size_block
        id_mask = config_diffusion.id_mask
        len_prompt = config_diffusion.len_prompt
        sorter = config_diffusion.klass_sorter()
        collector = config_diffusion.klass_collector()
        
        p_finalized = torch.zeros(x.shape, dtype=torch.float64, device=x.device)
        position_start = 0

        for id_block in range(num_blocks):
            position_end = position_start + len_prompt + (id_block+1) * size_block
            mask_mask_blk = x[:,position_start:position_end] == id_mask
            
            idx_denoising = torch.arange(position_start, position_end, dtype=torch.long).to(x.device)
            quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

            for step in range(step_per_block):
                x_denoising,  y_denoising= x[:, idx_denoising], y[:, idx_denoising]
                logits = model(x_denoising, idx_current=idx_denoising).logits
                snapshot = SimpleLogitsSnapshot(logits, x_denoising, y_denoising, id_mask)
                conf_snapshot = snapshot.transform_logits(collector)
                idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
                num_unmask = quota_helper.get_quota(step)
                idx_transform = idx_sorted_by_conf[:, :num_unmask]

                snapshot.materialize_by_idx_(idx_transform, conf_snapshot)
                snapshot.update_this(1, idx_transform, y=x).update_this(1, idx_transform, p_finalized=p_finalized)
                
            # end for step
        # end for block

        return p_finalized[:, len_prompt:]
    # end    
# end



class LLaDAYukaiHarness(LM):

    def __init__(self, *args, **kwargs):
        super().__init__()
        config = DiffusionConfig(
            id_model=kwargs['id_model'],
            len_prompt=kwargs['len_prompt'],
            len_target=kwargs['len_target'],
            num_blocks=kwargs['num_blocks'],
            num_unmask_per_step=kwargs['num_unmask_per_step'],
            id_mask=kwargs['id_mask'],
            size_batch=kwargs['size_batch'],
            device=kwargs['device'],
            klass_sorter=load_klass(f'tools_llada.{kwargs["klass_sorter"]}'),
            klass_collector=load_klass(f'tools_llada.{kwargs["klass_collector"]}'),
            klass_save_kv_previous=SaveKVPreviousPlugin_Disabled,
            klass_cache_past_kv=CachePastKVPlugin_Enabled,
            klass_cache_attn=CacheAttnPlugin_Disabled,
            klass_cache_vo=CacheVOPlugin_Disabled
        )


        self.tokenizer = self._init_tokenizer(config)
        self.model = self._init_model(config).eval()
        self.runmodel = RunModelSemi()

    # end

    def _init_tokenizer(self, config: DiffusionConfig):
        tokenizer = AutoTokenizer.from_pretrained(
            config.id_model,
            trust_remote_code=True
        )

        if tokenizer.padding_side != 'left':
            tokenizer.padding_side = 'left'
        # end

        assert tokenizer.pad_token_id != 126336
        return tokenizer
    # end

    def _init_model(self, config: DiffusionConfig):
        model = LLaDAModelLM.from_pretrained(
            config.id_model,
            trust_remote_code=True,
            torch_dtype=DTYPE_EVAL,
        ).to(config.device)

        return model
    # end


    def generate_until(self, requests: list[Instance]):
        pass
    # end

# end